# CNU Campus ChatBot — Colab 실행 노트북
**박연진 | 자연어처리 텀프로젝트**

## 실행 순서
```
[최초 1회]  셀 A → 런타임 재시작 → 셀 B → 셀 0 이후 순서대로
[이후 세션] 셀 0 (진입점) → 셀 3~7
```

| 셀 | 목적 | 실행 조건 |
|:---:|---|---|
| A | 패키지 설치 | **최초 1회만** → 런타임 재시작 |
| B | 최초 clone | **최초 1회만** |
| 0 | 진입점 (Drive + 경로 + 패키지 확인) | **매 세션마다** |
| 1 | 데이터 갱신 (TTL 크롤링) | 선택적 |
| 2 | ChromaDB 구축 | chroma_db 없을 때만 |
| 3 | 7B 모델 Drive 캐시 확인/저장 | 최초 또는 Drive 삭제 시 |
| 4 | 실행 전 체크리스트 | 선택적 |
| 5 | chatbot.sh (추론 + UI) | **핵심 실행** |
| 6 | chat_output.json 확인 | 검증용 |
| 7 | UI만 단독 실행 | 재실행/데모 |


In [1]:
!pip freeze > requirements.txt

In [3]:
!cat requirements.txt | grep torch

torch==2.11.0+cu128
torchao==0.10.0
torchaudio==2.11.0+cu128
torchcodec==0.11.0+cu128
torchdata==0.11.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision==0.26.0+cu128


In [4]:
import sys
import torch

print(sys.version)
print(torch.__version__)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
2.11.0+cu128


In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 A] 패키지 설치 — 최초 1회만 실행
# 완료 후 [런타임] > [런타임 다시 시작] 클릭
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Step 1: CUDA 버전 자동 감지
import subprocess, re
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
m = re.search(r"release (\d+)\.(\d+)", r.stdout)
if m:
    major, minor = int(m.group(1)), int(m.group(2))
    if major == 12 and minor <= 1:
        cuda_tag = "cu121"
    elif major == 12 and minor <= 4:
        cuda_tag = "cu124"
    else:                        # CUDA 12.5+ (12.8 등)
        cuda_tag = "cu124"       # torch 2.5.1 최고 호환 wheel
    print(f"CUDA {major}.{minor} 감지 -> wheel: {cuda_tag}")
else:
    cuda_tag = "cu124"           # nvcc 없는 환경 fallback
    print(f"CUDA 버전 감지 실패 -> fallback: {cuda_tag}")

# Step 2: torch 2.5.1 먼저 고정 설치
print("torch 2.5.1 설치 중...")
import sys
get_ipython().system(f"{sys.executable} -m pip install -q torch==2.5.1+{cuda_tag} torchvision==0.20.1+{cuda_tag} torchaudio==2.5.1+{cuda_tag} --index-url https://download.pytorch.org/whl/{cuda_tag}")

# Step 3: 나머지 패키지 설치
print("나머지 패키지 설치 중...")
get_ipython().system("pip install -q \n    "transformers>=4.40.0,<5.0.0" \n    "accelerate>=0.34.0" \n    "bitsandbytes>=0.43.3" \n    "datasets>=2.18.0" \n    scikit-learn \n    kiwipiepy \n    rank_bm25 \n    chromadb \n    "sentence-transformers==3.0.1" \n    "pytorch-lightning==2.4.0" \n    gradio \n    requests \n    beautifulsoup4 \n    tqdm")

# Step 4: torch 2.5.1 재설치 (sentence-transformers 등이 덮어쓸 수 있으므로 재고정)
print("torch 2.5.1 재고정 중...")
get_ipython().system(f"{sys.executable} -m pip install -q torch==2.5.1+{cuda_tag} torchvision==0.20.1+{cuda_tag} torchaudio==2.5.1+{cuda_tag} --index-url https://download.pytorch.org/whl/{cuda_tag}")

# ── 최종 검증 ──────────────────────────────────────────────
print()
print("=" * 50)
print("  최종 환경 검증")
print("=" * 50)

# Python 버전
import sys as _sys
pv = _sys.version_info
pv_str = f"{pv.major}.{pv.minor}.{pv.micro}"
if pv.major == 3 and pv.minor == 10 and pv.micro == 12:
    print(f"  OK  Python {pv_str}")
else:
    print(f"  WARN Python {pv_str} (과제 요구: 3.10.12 — Colab 런타임 고정값으로 변경 불가)")

# torch 버전
r2 = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    capture_output=True, text=True
)
installed = r2.stdout.strip()
if installed.startswith("2.5.1"):
    print(f"  OK  torch {installed}")
else:
    print(f"  WARN torch {installed} != 2.5.1 — 셀 A 재실행 후 런타임 재시작 필요")

# CUDA 사용 가능 여부
r3 = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(torch.cuda.is_available()); "
     "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')"],
    capture_output=True, text=True
)
cuda_lines = r3.stdout.strip().splitlines()
cuda_ok = len(cuda_lines) >= 1 and cuda_lines[0].strip() == "True"
gpu_name = cuda_lines[1] if len(cuda_lines) >= 2 else "N/A"
if cuda_ok:
    print(f"  OK  CUDA 사용 가능 ({gpu_name})")
else:
    print("  WARN CUDA 사용 불가 — [런타임 유형 변경] > T4 GPU 선택 필요")

# pytorch-lightning 버전
r4 = subprocess.run(
    [sys.executable, "-c", "import pytorch_lightning; print(pytorch_lightning.__version__)"],
    capture_output=True, text=True
)
pl_ver = r4.stdout.strip()
if pl_ver == "2.4.0":
    print(f"  OK  pytorch-lightning {pl_ver}")
else:
    print(f"  WARN pytorch-lightning {pl_ver} != 2.4.0")

print()
print("설치 완료!")
print("→ [런타임] > [런타임 다시 시작] 클릭 후 셀 B 실행")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 B] 레포 Clone — 최초 1회만 실행
# 이미 clone 됐으면 건너뛰고 셀 0 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os

PROJECT = '/content/drive/MyDrive/NLP_Term_Project'

if os.path.exists(PROJECT):
    print(f"이미 존재: {PROJECT}")
    print("→ Clone 건너뜀. 셀 0 실행으로 이동하세요.")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive
    !git clone https://github.com/capyjin/NLP_Term_Project.git
    print("Clone 완료!")
    !ls NLP_Term_Project/

이미 존재: /content/drive/MyDrive/NLP_Term_Project
→ Clone 건너뜀. 셀 0 실행으로 이동하세요.


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 0] 진입점 — 매 세션마다 가장 먼저 실행
# Drive 마운트 + 경로 설정 + git pull + 패키지 확인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount("/content/drive")

import os, sys, importlib
PROJECT = "/content/drive/MyDrive/NLP_Term_Project"
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print("작업 경로:", os.getcwd())

# Python 버전 확인
py = sys.version_info
py_str = f"{py.major}.{py.minor}.{py.micro}"
if py.major == 3 and py.minor == 10:
    print(f"OK  Python {py_str}")
else:
    print(f"WARN Python {py_str} (과제 요구: 3.10.12)")

# GPU + torch 버전 확인
import torch
torch_ver = torch.__version__
if torch_ver.startswith("2.5.1"):
    print(f"OK  torch {torch_ver}")
else:
    print(f"WARN torch {torch_ver} != 2.5.1 — 셀 A 재실행 후 런타임 재시작 필요")

if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {gb:.1f} GB")
else:
    print("CPU 환경 — [런타임] > [런타임 유형 변경] > T4 GPU 선택 필요")

# 최신 코드 Pull
get_ipython().system("git pull origin main")

# 패키지 확인
print()
for pkg in ["transformers", "bitsandbytes", "kiwipiepy", "rank_bm25", "chromadb", "gradio", "sentence_transformers"]:
    try:
        importlib.import_module(pkg)
        print(f"  OK {pkg}")
    except ImportError:
        print(f"  !! {pkg} 없음 -> 셀 A 실행 후 런타임 재시작 필요")

Mounted at /content/drive
작업 경로: /content/drive/MyDrive/NLP_Term_Project
GPU: Tesla T4 | VRAM: 15.6 GB
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 15 (delta 11), reused 15 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 8.73 KiB | 1024 bytes/s, done.
From https://github.com/capyjin/NLP_Term_Project
 * branch            main       -> FETCH_HEAD
   71919ae..3518871  main       -> origin/main
Updating 71919ae..3518871
Fast-forward
 data/raw/academic_calendar.json           | 421 ++++++++++++++++++++++++++++++
 scripts/refresh_data.py                   |  34 ++-
 src/chatbot_router.py                     |  73 ++++--
 src/chatbot_ui.py                         |   3 +-
 src/crawling/academic_calendar_crawler.py | 208 +++++++++++++++
 src/handlers/academic_calendar_handler.py | 270 +++++++++++++++++++
 src/realtime_model.py                     |  17 +-
 7 files changed, 9

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 1] 데이터 갱신 — 식단/셔틀/공지 크롤링
# TTL: 식단 6h / 셔틀 24h / 공지 6h
# 이미 최신이면 건너뛰어도 됨
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!python scripts/refresh_data.py
# 특정 항목만 갱신하려면:
# !python scripts/refresh_data.py --meal      # 식단만
# !python scripts/refresh_data.py --shuttle   # 셔틀만
# !python scripts/refresh_data.py --notice    # 공지/장학만


  CNU ChatBot — 데이터 갱신

  식단 데이터 갱신
  → 식단 크롤링 중... (meal_crawler.py)
  ✓ 식단 갱신 완료
  ✓ meal_menu.json: 3건 저장됨

  셔틀버스 데이터 갱신
  → 셔틀버스 크롤링 중... (shuttle_crawler.py)
  ✓ 셔틀버스 갱신 완료

  공지/장학 데이터 갱신
  → 공지/장학 크롤링 중... (cnu_crawler.py)
  ⚠ 공지/장학 크롤링 실패 — 기존 파일 유지
     ModuleNotFoundError: No module named 'selenium'

  학사일정 데이터 갱신
  ✓ 학사일정 파일 유효 (갱신된 지 0.0시간) — 스킵

  벡터 DB 재구축
  → FAQ 재삽입 중...
  ✓ FAQ 재삽입 완료
  → ChromaDB 임베딩 중... (~5~10분)
  ✓ 벡터 DB 재구축 완료

  완료
  ✓ 데이터 갱신 완료. UI 실행: python src/chatbot_ui.py


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 2] ChromaDB 구축 — chroma_db 없을 때만 실행
# 있으면 건너뜀 (Drive에 영구 저장됨)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, shutil, json

# FAQ inject 상태 확인
chunks_path = 'data/processed/chunks.json'
if os.path.exists(chunks_path):
    with open(chunks_path, encoding='utf-8') as f:
        chunks = json.load(f)
    faq_count = sum(1 for c in chunks if c.get('source_type') == 'faq_manual')
    print(f"chunks.json: {len(chunks)}개 (FAQ: {faq_count}개)")
    if faq_count < 23:
        print("FAQ 부족 -> inject 실행")
        !python scripts/inject_faq.py
    else:
        print("FAQ OK")

# chroma_db 구축
if os.path.exists('chroma_db') and len(os.listdir('chroma_db')) > 0:
    print("chroma_db 이미 존재 -> 건너뜀")
    print("  (재구축 필요 시: shutil.rmtree('chroma_db') 후 재실행)")
else:
    print("chroma_db 구축 시작...")
    !python src/vectordb/build_db.py
    if os.path.exists('chroma_db'):
        print("chroma_db 구축 완료!")
    else:
        print("구축 실패 - 위 오류 확인 필요")

chunks.json: 205개 (FAQ: 47개)
FAQ OK
chroma_db 이미 존재 -> 건너뜀
  (재구축 필요 시: shutil.rmtree('chroma_db') 후 재실행)


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 3] Qwen2.5-3B Drive 캐시 확인/저장
# Drive에 없으면 HuggingFace에서 다운로드 (~3GB, 3~5분)
# 이미 저장됐으면 즉시 완료
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DRIVE_3B = '/content/drive/MyDrive/models/qwen2.5-3b-4bit'
HF_3B    = 'Qwen/Qwen2.5-3B-Instruct'

if os.path.exists(DRIVE_3B) and os.path.exists(os.path.join(DRIVE_3B, 'config.json')):
    print(f"3B 모델 Drive 캐시 확인: {DRIVE_3B}")
    print("  -> 이미 저장됨. chatbot.sh에서 Drive 경로로 빠르게 로드됩니다.")
else:
    print(f"Drive 캐시 없음 -> {HF_3B} 다운로드 시작 (~3GB, 3~5분)")
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    model = AutoModelForCausalLM.from_pretrained(
        HF_3B,
        quantization_config=quant,
        device_map={'': 0},        # GPU 0 강제 (CPU offload 방지)
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(HF_3B)
    os.makedirs(DRIVE_3B, exist_ok=True)
    model.save_pretrained(DRIVE_3B)
    tokenizer.save_pretrained(DRIVE_3B)
    print(f"Drive 저장 완료: {DRIVE_3B}")
    del model  # VRAM 해제
    import gc; gc.collect(); torch.cuda.empty_cache()
    print("VRAM 해제 완료. chatbot.sh 실행 준비됨.")

3B 모델 Drive 캐시 확인: /content/drive/MyDrive/models/qwen2.5-3b-4bit
  -> 이미 저장됨. chatbot.sh에서 Drive 경로로 빠르게 로드됩니다.


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 4] 실행 전 체크리스트
# 모든 항목이 OK여야 chatbot.sh 정상 실행 가능
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, json

checks = {
    "chroma_db 구축됨": os.path.exists('chroma_db') and len(os.listdir('chroma_db')) > 0,
    "chunks.json 존재": os.path.exists('data/processed/chunks.json'),
    "meal_menu.json 존재": os.path.exists('data/raw/meal_menu.json'),
    "shuttle_bus.json 존재": os.path.exists('data/raw/shuttle_bus.json'),
    "test_chat.json 존재": os.path.exists('data/test_chat.json'),
    "7B Drive 캐시 존재": os.path.exists('/content/drive/MyDrive/models/qwen2.5-7b-4bit/config.json'),
    "chatbot.sh 존재": os.path.exists('chatbot.sh'),
    "src/chatbot_ui.py 존재": os.path.exists('src/chatbot_ui.py'),
}

all_ok = True
for name, ok in checks.items():
    status = 'OK' if ok else '!!'
    print(f"  [{status}] {name}")
    if not ok: all_ok = False

# test_chat.json 없으면 placeholder 생성
if not os.path.exists('data/test_chat.json'):
    placeholder = [
        {"id": 1, "question": "졸업 학점이 몇 점인가요?"},
        {"id": 2, "question": "장학금 신청은 어디서 해?"},
        {"id": 3, "question": "수강신청 변경 기간은 언제야?"},
        {"id": 4, "question": "오늘 학식 뭐 나와?"},
        {"id": 5, "question": "셔틀버스 시간표 알려줘"},
    ]
    with open('data/test_chat.json', 'w', encoding='utf-8') as f:
        json.dump(placeholder, f, ensure_ascii=False, indent=2)
    print("  -> test_chat.json placeholder 생성 완료")

print()
if all_ok:
    print("모든 준비 완료 -> 셀 5 (chatbot.sh) 실행하세요!")
else:
    print("!! 위 항목 확인 후 해당 셀 재실행 필요")

  [OK] chroma_db 구축됨
  [OK] chunks.json 존재
  [OK] meal_menu.json 존재
  [OK] shuttle_bus.json 존재
  [OK] test_chat.json 존재
  [OK] 7B Drive 캐시 존재
  [OK] chatbot.sh 존재
  [OK] src/chatbot_ui.py 존재

모든 준비 완료 -> 셀 5 (chatbot.sh) 실행하세요!


In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 5] chatbot.sh 실행
# 전체 파이프라인: 크롤 -> 추론 -> UI
#   [3/4] test_chat.json -> outputs/chat_output.json 생성
#   [4/4] Gradio UI 실행 -> public URL 출력
# Drive 캐시 있으면 ~30초, 없으면 ~30분 소요
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!bash chatbot.sh

  CNU Campus ChatBot

[2/4] 식단/셔틀버스 데이터 업데이트 중...
[meal_crawler] 주간 식단 크롤링 시작
  2026-06-08 (월): 6건
  2026-06-09 (화): 6건
  2026-06-10 (수): 6건
  2026-06-11 (목): 6건
  2026-06-12 (금): 5건
  2026-06-13: 메뉴 없음 (운영 없는 날)
[meal_crawler] 저장 완료: /content/drive/MyDrive/NLP_Term_Project/data/raw/meal_menu.json (총 29건)
[shuttle_crawler] 크롤링 시작: https://plus.cnu.ac.kr/html/kr/sub05/sub05_050403.html
  [parser] 크롤링 성공: 2노선
[shuttle_crawler] 저장 완료: /content/drive/MyDrive/NLP_Term_Project/data/raw/shuttle_bus.json (2노선)

[3/4] 챗봇 일괄 추론: data/test_chat.json → outputs/chat_output.json
입력 로드: 3건 (/content/drive/MyDrive/NLP_Term_Project/data/test_chat.json)
ChatBot 초기화 중... (첫 실행 시 모델 다운로드 ~10분)
Loading weights: 100% 391/391 [00:00<00:00, 20994.88it/s]
BM25 인덱스 구축 완료: 205청크 (크롤링 158건 + FAQ 47건)
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `qua

In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 6] chat_output.json 결과 확인
# chatbot.sh 완료 후 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, os

path = 'outputs/chat_output.json'
if os.path.exists(path):
    with open(path, encoding='utf-8') as f:
        result = json.load(f)
    print(f"chat_output.json 생성 완료 - {len(result)}건\n")
    for i, item in enumerate(result, 1):
        q = item.get('question', item.get('user', ''))
        a = item.get('answer', item.get('model', ''))
        src = item.get('source', '')
        print(f"Q{i}: {q}")
        print(f"A{i}: {a[:120]}..." if len(a) > 120 else f"A{i}: {a}")
        if src:
            print(f"   [경로: {src}]")
        print()
else:
    print("chat_output.json 없음 - chatbot.sh 실행 중 오류 확인")

chat_output.json 생성 완료 - 3건

Q1: 졸업 학점이 몇 점인가요?
A1: 충남대학교 졸업요건은 학과, 입학연도, 전공(단일/복수/부전공) 여부에 따라 달라집니다.

일반적인 졸업 요건 구성:
  • 총 이수학점: 130~140학점 내외 (학과별 상이)
  • 전공 이수: 전공필수 전부 +...

Q2: 장학금 신청은 어디서 해?
A2: 장학금 신청은 다음과 같은 곳에서 진행됩니다:

1. **국가장학금**: 
   - 한국장학재단 홈페이지(www.kosaf.go.kr)에서 신청합니다.
   - 신청 절차는 아래와 같습니다:
     1. 한국장학재...

Q3: 수강신청 변경 기간은 언제야?
A3: 수강신청 및 관련 학사일정은 학기마다 다르게 운영됩니다.

수강신청 주요 안내:
  • 예비 수강신청: 학기 시작 2~3주 전 (희망 과목 선점)
  • 본 수강신청: 학기 시작 1~2주 전 (확정)
  • 수강정정...



In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [셀 7] UI 단독 실행 (데모/재실행용)
# chatbot.sh 없이 UI만 띄울 때 사용
# 이미 chatbot.sh로 UI가 실행 중이면 건너뜀
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 옵션 1: 데이터 갱신 + UI
!bash run_ui.sh

# 옵션 2: 데이터 갱신 건너뛰고 UI만
# !bash run_ui.sh --skip

# 옵션 3: UI 직접 실행
# !python src/chatbot_ui.py